# ⚠️ IMPORTANT — THIS NOTEBOOK DOES NOT RUN AS-IS

> **This notebook is a template/reference and will NOT work out of the box.**
> Before running anything, you MUST review and update the configuration below for your own environment.

**You need to:**

1. **Specify paths to the model weights.** The paths in this notebook point to a previous setup and will not exist on your machine. Update every path that loads or saves model weights / checkpoints (e.g. the `pathTo(...)` base path `"/content/drive/My Drive/multilingual_ner (1)/"`).
2. **Change the model names if needed.** Update the Hugging Face model IDs / local model names to the ones you actually intend to use.
3. **Point to all required input correctly.** Fix the paths to datasets, NER input files, label mappings, and any other input the cells read from. Nothing will load until these point to real, existing files.
4. **Check the runtime/environment.** This notebook assumes Google Colab + Google Drive (`drive.mount(...)`) and a CUDA GPU. Adjust the install, mount, and device cells if you are running elsewhere.

**In short: go cell by cell and replace placeholder paths, model names, and input locations with values that are valid for your setup. Running the cells unmodified will fail.**


In [ ]:
# !pip install evaluate seqeval transformers sentence-transformers trl peft accelerate
!pip install torch torchvision datasets evaluate seqeval transformers==4.45.2 sentence-transformers==3.1.1 trl peft==0.14.0 accelerate bitsandbytes googletrans
# !pip install bitsandbytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import sys
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
import transformers
import bitsandbytes
import torch
import pandas as pd
# from bitsandbytes import BitsAndBytesConfig
from transformers import BitsAndBytesConfig
import torch
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import pandas as pd
import traceback
from googletrans import Translator
from transformers import AutoTokenizer, AutoModelForTokenClassification,pipeline






In [ ]:
# Translate with googletrans to es
from googletrans import Translator
translator = Translator()

# !ls -l "/content/drive/My Drive/multilingual_ner (1)/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.backends.mps.is_available()
torch.cuda.is_available()

id2label = {0: 'Designation', 1: 'Companies worked at', 2: 'Name', 3: 'Years of Experience', 4: 'Soft Skills', 5: 'Email Address', 6: 'Graduation Year', 7: 'Degree', 8: 'Tech Tools', 9: 'Location', 10: 'Job Specific Skills', 11: 'College Name', 12: 'O'}
label2id = {'Designation': 0, 'Companies worked at': 1, 'Name': 2, 'Years of Experience': 3, 'Soft Skills': 4, 'Email Address': 5, 'Graduation Year': 6, 'Degree': 7, 'Tech Tools': 8, 'Location': 9, 'Job Specific Skills': 10, 'College Name': 11, 'O': 12}
all_possible_labels = {'Designation', 'Companies worked at', 'Name', 'Years of Experience', 'Soft Skills', 'Email Address', 'Graduation Year', 'Degree', 'Tech Tools', 'Location', 'Job Specific Skills', 'College Name'}


# pathTo = lambda x: "./" + x
pathTo = lambda x: "/content/drive/My Drive/multilingual_ner (1)/" + x
print(pathTo("."))


MAX_LENGTH=3000

In [ ]:
# -*- coding: utf-8 -*-

from copy import deepcopy

from transformers.models.llama.modeling_llama import *
from transformers.modeling_outputs import TokenClassifierOutput


import torch
import torch.nn as nn
from transformers.models.llama.modeling_llama import LlamaDecoderLayer,LlamaRMSNorm
from peft import get_peft_model, LoraConfig, TaskType
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

# Copied from transformers.models.bart.modeling_bart._make_causal_mask
def _make_causal_mask(
    input_ids_shape, dtype, device, past_key_values_length: int = 0
):
    """
    Make causal mask used for bi-directional self-attention.
    """
    bsz, tgt_len = input_ids_shape
    mask = torch.full((tgt_len, tgt_len), torch.finfo(dtype).min, device=device)
    mask_cond = torch.arange(mask.size(-1), device=device)
    mask.masked_fill_(mask_cond < (mask_cond + 1).view(mask.size(-1), 1), 0)
    mask = mask.to(dtype)

    if past_key_values_length > 0:
        mask = torch.cat([torch.zeros(tgt_len, past_key_values_length, dtype=dtype, device=device), mask], dim=-1)
    return mask[None, None, :, :].expand(bsz, 1, tgt_len, tgt_len + past_key_values_length)



# Copied from transformers.models.bart.modeling_bart._expand_mask
def _expand_mask(mask, dtype, tgt_len = None):
    """
    Expands attention_mask from `[bsz, seq_len]` to `[bsz, 1, tgt_seq_len, src_seq_len]`.
    """
    bsz, src_len = mask.size()
    tgt_len = tgt_len if tgt_len is not None else src_len

    expanded_mask = mask[:, None, None, :].expand(bsz, 1, tgt_len, src_len).to(dtype)

    inverted_mask = 1.0 - expanded_mask

    return inverted_mask.masked_fill(inverted_mask.to(torch.bool), torch.finfo(dtype).min)


class UnmaskingLlamaModel(LlamaPreTrainedModel):

    def __init__(self, config):
        super().__init__(config)
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, self.padding_idx)
        self.layers = nn.ModuleList([LlamaDecoderLayer(config, _) for _ in range(config.num_hidden_layers)])
        self.norm = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        self.gradient_checkpointing = False
        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self):
        return self.embed_tokens

    def set_input_embeddings(self, value):
        self.embed_tokens = value

    # Copied from transformers.models.bart.modeling_bart.BartDecoder._prepare_decoder_attention_mask
    def _prepare_decoder_attention_mask(self, attention_mask, input_shape, inputs_embeds, past_key_values_length):
        # create causal mask
        # [bsz, seq_len] -> [bsz, 1, tgt_seq_len, src_seq_len]
        combined_attention_mask = None
        if input_shape[-1] > 1:
            combined_attention_mask = _make_causal_mask(
                input_shape,
                inputs_embeds.dtype,
                device=inputs_embeds.device,
                past_key_values_length=past_key_values_length,
            )
        if attention_mask is not None:
            # [bsz, seq_len] -> [bsz, 1, tgt_seq_len, src_seq_len]
            expanded_attn_mask = _expand_mask(attention_mask, inputs_embeds.dtype, tgt_len=input_shape[-1]).to(
                inputs_embeds.device
            )
            combined_attention_mask = (
                expanded_attn_mask if combined_attention_mask is None else expanded_attn_mask + combined_attention_mask
            )

        return combined_attention_mask

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        use_cache = use_cache if use_cache is not None else self.config.use_cache

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        # retrieve input_ids and inputs_embeds
        if input_ids is not None and inputs_embeds is not None:
            raise ValueError("You cannot specify both decoder_input_ids and decoder_inputs_embeds at the same time")
        elif input_ids is not None:
            batch_size, seq_length = input_ids.shape
        elif inputs_embeds is not None:
            batch_size, seq_length, _ = inputs_embeds.shape
        else:
            raise ValueError("You have to specify either decoder_input_ids or decoder_inputs_embeds")

        seq_length_with_past = seq_length
        past_key_values_length = 0

        if past_key_values is not None:
            past_key_values_length = past_key_values[0][0].shape[2]
            seq_length_with_past = seq_length_with_past + past_key_values_length

        if position_ids is None:
            device = input_ids.device if input_ids is not None else inputs_embeds.device
            position_ids = torch.arange(
                past_key_values_length, seq_length + past_key_values_length, dtype=torch.long, device=device
            )
            position_ids = position_ids.unsqueeze(0).view(-1, seq_length)
        else:
            position_ids = position_ids.view(-1, seq_length).long()

        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        # embed positions
        if attention_mask is None:
            attention_mask = torch.ones(
                (batch_size, seq_length_with_past), dtype=torch.bool, device=inputs_embeds.device
            )
        # causal mask
        '''
        attention_mask = self._prepare_decoder_attention_mask(
            attention_mask, (batch_size, seq_length), inputs_embeds, past_key_values_length
        )
        print('unmasking attention mask:')
        print(attention_mask)
        '''
        # remove causal mask
        attention_mask = torch.zeros(
            (batch_size, 1, seq_length, seq_length), device=inputs_embeds.device
        )

        hidden_states = inputs_embeds

        if self.gradient_checkpointing and self.training:
            if use_cache:
                logger.warning_once(
                    "`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`..."
                )
                use_cache = False

        # decoder layers
        all_hidden_states = () if output_hidden_states else None
        all_self_attns = () if output_attentions else None
        next_decoder_cache = () if use_cache else None

        for idx, decoder_layer in enumerate(self.layers):
            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            past_key_value = past_key_values[idx] if past_key_values is not None else None

            if self.gradient_checkpointing and self.training:

                def create_custom_forward(module):
                    def custom_forward(*inputs):
                        # None for past_key_value
                        return module(*inputs, past_key_value, output_attentions)

                    return custom_forward

                layer_outputs = torch.utils.checkpoint.checkpoint(
                    create_custom_forward(decoder_layer),
                    hidden_states,
                    attention_mask,
                    position_ids,
                )
            else:
                layer_outputs = decoder_layer(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_value,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                )

            hidden_states = layer_outputs[0]

            if use_cache:
                next_decoder_cache += (layer_outputs[2 if output_attentions else 1],)

            if output_attentions:
                all_self_attns += (layer_outputs[1],)

        hidden_states = self.norm(hidden_states)

        # add hidden states from the last decoder layer
        if output_hidden_states:
            all_hidden_states += (hidden_states,)

        next_cache = next_decoder_cache if use_cache else None
        if not return_dict:
            return tuple(v for v in [hidden_states, next_cache, all_hidden_states, all_self_attns] if v is not None)
        return BaseModelOutputWithPast(
            last_hidden_state=hidden_states,
            past_key_values=next_cache,
            hidden_states=all_hidden_states,
            attentions=all_self_attns,
        )



class UnmaskingLlamaForTokenClassification(LlamaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.model = UnmaskingLlamaModel(config)
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()

    def get_input_embeddings(self):
        return self.model.embed_tokens

    def set_input_embeddings(self, value):
        self.model.embed_tokens = value

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        labels=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.model(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]

        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )




In [ ]:
# wikineural: LOAD THE OTHER MODEL for



babel_tokenizer = AutoTokenizer.from_pretrained("Babelscape/wikineural-multilingual-ner")
bable_model = AutoModelForTokenClassification.from_pretrained("Babelscape/wikineural-multilingual-ner")



In [ ]:


# !cp -r "/scontent/drive/My Drive/unmasked-exp/my_awesome_ds_model-final-final-final-best" "$path/trained"
model_id = pathTo("trained/my_awesome_ds_model-final-final-final-best")
print(model_id)
bilora_tokenizer = AutoTokenizer.from_pretrained(model_id)
import torch.nn as nn
bilora_model = UnmaskingLlamaForTokenClassification.from_pretrained(
    model_id, num_labels=len(label2id), id2label=id2label, label2id=label2id,

)
bilora_model = bilora_model.to('cuda')


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, pipeline, BitsAndBytesConfig

# --- Configuration ---
USE_4BIT = False  # Set to True to load with 4-bit quantization (requires bitsandbytes)
# Set to a local path to load a pre-trained/fine-tuned model, or None to use HF hub
MODEL_PATH = None  # e.g. "/path/to/local/llama-3.1-8b-4bit"

llama_31_model_id = "meta-llama/Llama-3.1-8B-Instruct"
model_source = MODEL_PATH if MODEL_PATH else llama_31_model_id

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model_kwargs = dict(quantization_config=bnb_config, device_map="auto")
    pipeline_dtype = torch.bfloat16
else:
    # Full precision (bfloat16)
    model_kwargs = dict(torch_dtype=torch.bfloat16, device_map=device)
    pipeline_dtype = torch.bfloat16

llama_31_tokenizer = AutoTokenizer.from_pretrained(model_source)
llama_31 = AutoModelForCausalLM.from_pretrained(model_source, **model_kwargs)

if not USE_4BIT:
    llama_31 = llama_31.to(device)

llama_31_pipeline = pipeline(
    "text-generation",
    model=llama_31,
    tokenizer=llama_31_tokenizer,
    torch_dtype=pipeline_dtype,
    device_map="auto" if USE_4BIT else device,
    max_length=MAX_LENGTH,
)


In [ ]:

def translate_with_llama32(text, source_lang='es', target_lang='en'):
    model = llama_31
    tokenizer = llama_31_tokenizer
    """Translate text using Llama-3.2-3B"""
    # This is a common pattern for instruction-following models.
    prompt = f"Translate the following {source_lang} text to {target_lang}. Only provide the translation, no explanations, keep entities names as is, whether it's names, tools:\n\n{text}, Translation:"


    messages = [
        # {
        #     "role": "system",
        #     "content": "You are a helpful AI assistant. Answer the user's questions clearly and concisely."
        # },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 4. Format Input with Chat Template
    # This applies the special tokens like <|start_header_id|> automatically
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)



    # 6. Generate Output with Ollama-equivalent Parameters
    outputs = model.generate(
        input_ids,
        max_new_tokens=2048,      # Limits response length
        temperature=0.8,          # Controls creativity (Ollama default ~0.8)
        top_p=0.9,                # Nucleus sampling
        top_k=40,                 # Limits vocabulary choices
        repetition_penalty=1.1,   # Prevents repeating phrases
        do_sample=True,           # Required for temp/top_p to work
        # streamer=streamer         # Print to console as it generates
    )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

    print(prompt)

    output = llama_32_pipeline(prompt)
    after_translation = output[0]['generated_text'].split('Translation:')[1]
    return after_translation


    processed = llama_32_processor.apply_chat_template(messages,
    add_generation_prompt=True, tokenize=False)
    inputs = llama_32_processor(processed, return_tensors="pt")


    # inputs = llama_32_tokenizer(prompt, return_tensors="pt").to(llama_32.device)

    outputs = llama_32.generate(
        **inputs,
        max_new_tokens=1000,
        # temperature=0.3,
        # do_sample=True,
        # pad_token_id=tokenizer.eos_token_id
    )

    translation = llama_32_processor.decode(outputs[0], skip_special_tokens=True)

    return translation

# Assuming sampleCV_ES is already defined from cell 3-3j1exer-Ft
print("Translating sampleCV_ES using Llama-3.2-3B...")
sampleCV_EN_FROM_ES_LLAMA32 = translate_with_llama32(sampleCV_ES, source_lang='spanish', target_lang='english')
print(sampleCV_EN_FROM_ES_LLAMA32)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=bilora_tokenizer)
# Create mock trainer just to run predicate
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

lora_r = 12

# peft_config = LoraConfig(task_type=TaskType.TOKEN_CLS, inference_mode=False, r=lora_r, lora_alpha=32, lora_dropout=0.1)
# bilora_model = get_peft_model(bilora_model, peft_config)
# bilora_model.print_trainable_parameters()

trainer = Trainer(
    model=bilora_model,
    train_dataset=[],
    eval_dataset=[],
    tokenizer=bilora_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# Define a tokenization function for raw text (without label alignment)
def tokenize_raw_text(examples):
    # Assuming tokenizer and max_length are defined in a previous cell
    return bilora_tokenizer(examples["text"], padding='longest', max_length=MAX_LENGTH, truncation=True, return_offsets_mapping=True)
def bilora_ner(text):
  llama_dataset_raw = Dataset.from_list([{'text': text} for text in [text]])
  tokenized_llama_ds = llama_dataset_raw.map(tokenize_raw_text, batched=True)

  # print("Tokenized Llama translated dataset created:")
  # print(tokenized_llama_ds)
  with torch.autocast("cuda"):

    # --- Run predictions on the test dataset ---
    results = trainer.predict(tokenized_llama_ds)

  predictions, label_ids, metrics = results.predictions, results.label_ids, results.metrics
  # Convert raw logits to label indices.
  predictions = np.argmax(predictions, axis=2)

  print(tokenized_llama_ds)
      # predictions = torch.argmax(outputs.logits, axis=2)
  file_content = []
  for idx, sample in enumerate(predictions):
    print(idx)
    ners = []
    for i in sample:
      # print(i)
      ners.append(id2label[i.item()])
    tokens = bilora_tokenizer.convert_ids_to_tokens(tokenized_llama_ds[idx]['input_ids'])
    offsets = (tokenized_llama_ds[idx]['offset_mapping'])
    print(len(offsets), len(tokens))
  return ners, tokens,offsets


sample_cv_bilora_ner_from_source_english = bilora_ner(sampleCV)
sample_cv_bilora_ner_from_google_spanish = bilora_ner(sampleCV_ES)
sample_cv_bilora_ner_from_llama_english = bilora_ner(sampleCV_EN_FROM_ES_LLAMA32)
print(sampleCV)
print(sample_cv_bilora_ner_from_source_english)
print(sampleCV_ES)
print(sample_cv_bilora_ner_from_google_spanish)
print(sampleCV_EN_FROM_ES_LLAMA32)
print(sample_cv_bilora_ner_from_llama_english)

In [ ]:
#
# with open('/content/drive/MyDrive/Colab Notebooks/CVNER/train.json', 'r') as file:
#     training_json = json.load(file)

import pandas as pd




all_possible_labels = set()
def load_set(name):
  global all_possible_labels
  training_json = pd.read_json(path_or_buf=pathTo(name), lines=True).to_dict(orient='records')
  ret = []
  for i in range(len(training_json)):
    element = training_json[i]

    text = element['text']
    labels = element['labels']
    text = text.replace('\\n','  ')
    text = text.replace('\n',' ')
    # print(text)
    newText=""
    lastEnd = 0
    # Order labels by starting index
    labels = sorted(labels, key=lambda x: x[0])



    parts = []
    partsEntities =[]

    for label in labels:
      start = label[0]
      end = label[1]
      label_name = label[2]
      all_possible_labels.add(label_name)
      ###########################################
      # Split to Parts where each part is the whole sentence or entity
      ###########################################
      # parts.extend([text[lastEnd:start],text[start:end]])
      # partsEntities.extend(['O',label_name])

      ###########################################
      # Split to parts where each part is a word
      ###########################################

      textPart = text[lastEnd:start]
      textParts = textPart.split(' ')
      textParts = list(filter(lambda x:x!='',textParts))
      parts.extend(textParts)
      partsEntities.extend(['O']*len(textParts))

      textPart = text[start:end]
      textParts = textPart.split(' ')
      textParts = list(filter(lambda x:x!='', textParts))
      parts.extend(textParts)
      partsEntities.extend([label_name]*len(textParts))




      # newText+=text[lastEnd:start]+"<"+label_name+">"+text[start:end]+"</"+label_name+">"
      lastEnd = end
    ret.append({'tokens':parts,'ner_tags':partsEntities})
  return ret


def load_dataset():
  ret = {}
  for split_name in ['train', 'test']:
      data = load_set(split_name+'.json')
      ret[split_name] = Dataset.from_list(data)

  return DatasetDict(ret)

dataset = load_dataset()
print(all_possible_labels)

In [ ]:
def tokenize_and_align_labels(examples):
      tokenized_inputs = bilora_tokenizer(examples["tokens"], is_split_into_words=True, padding='longest', max_length=MAX_LENGTH, truncation=True)

      labels = []
      for i, label in enumerate(examples[f"ner_tags"]):
          word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
          previous_word_idx = None
          label_ids = []
          for word_idx in word_ids:  # Set the special tokens to -100.
              if word_idx is None:
                  label_ids.append(-100)
              elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                  # Get Id from label
                  # label_id = label2id[]
                  # print(word_idx, label[word_idx])
                  id = label2id[label[word_idx]]
                  label_ids.append(id)
              else:
                  label_ids.append(-100)
              previous_word_idx = word_idx
          labels.append(label_ids)

      tokenized_inputs["labels"] = labels


In [ ]:




# Don't do this unless you want to translate with google from scratch








import time

translator = Translator()
destination_language = 'es'

async def translate_dataset_split(split_name):
    """Translate a dataset split to English"""
    translated_data = []
    original_data = dataset[split_name]

    for idx, example in enumerate(original_data):
        tokens = example['tokens']
        ner_tags = example['ner_tags']

        # Join tokens to form the original text
        text = ' '.join(tokens)

        try:
            # Translate to English
            translated = await translator.translate(text, dest=destination_language)
            translated_text = translated.text
            translated_data.append(translated_text)
            # Add small delay to avoid rate limiting
            time.sleep(0.1)

        except Exception as e:
            print(f"Error translating example {idx} in {split_name}: {e}")
            # Keep original if translation fails
            translated_data.append({
                'tokens': tokens,
                'ner_tags': ner_tags
            })

    return translated_data

# Translate both train and test splits
# print("Translating train split...")
# translated_train = translate_dataset_split('train')

print("Translating test split...")
translated_test = await translate_dataset_split('test')

# Create new translated dataset
translated_dataset = DatasetDict({
    # 'train': Dataset.from_list(translated_train),
    # 'test': Dataset.from_list(translated_test)
})

print(f"\nTranslation complete!")
# print(f"Translated train size: {len(translated_dataset['train'])}")
print(f"Translated test size: {len(translated_test)}")

# Save translated dataset locally
# translated_dataset['train'].to_json(pathTo('translated_train.json'))
# translated_test.to_json(pathTo('translated_test.json'))
json.dumps(translated_test)

with open(pathTo('translated_test.json'), "w") as final:
	json.dump(translated_test, final)

In [ ]:
import json

# Load the translated_test.json file
with open(pathTo('translated_test.json'), 'r') as f:
    translated_test = json.load(f)

# Print the first few entries to verify
print("Loaded translated_test.json (first 3 entries):")
for i, entry in enumerate(translated_test[:3]):
    print(f"Entry {i+1}: {entry}")

print(f"Total entries loaded: {len(translated_test)}")

In [ ]:
#



# Do this only unless you want to extract ner using wiki

# Load translated_dataset from saved files



translated_dataset = DatasetDict({
    # 'train': Dataset.from_pandas(pd.read_json(path_or_buf=pathTo('translated_train.json'), lines=True)),
    'test': Dataset.from_pandas(pd.read_json(path_or_buf=pathTo('translated_test.json'), lines=True))
})
# Load from Google translated and extract NER using Multilingual NER multilingual_ner_bable
def extract_ner_with_multilingual_model(split_name):
    """Extract NER from translated dataset using multilingual model"""
    ner_extracted_data = []
    translated_data = translated_dataset[split_name]

    # Create NER pipeline with the multilingual model
    multilingual_ner_pipeline = pipeline("ner", model=multilingual_ner_bable, tokenizer=tokenizer, grouped_entities=True)

    for  idx in (translated_data[0]):
        example = translated_data[0][idx]
        if(idx == 0):
            print(example)
        text = example
        tokens = text.split(' ')

        # Join tokens to form text
        text = ' '.join(tokens)

        # Extract NER using multilingual model
        ner_results = multilingual_ner_pipeline(text)

        if(idx == 0):
          print(ner_results)
        # Initialize all tokens with 'O' label
        ner_tags = ['O'] * len(tokens)

        # Map NER results to tokens
        for entity in ner_results:
            entity_text = entity['word'].strip()
            entity_label = entity['entity_group']

            # Find matching tokens
            for i, token in enumerate(tokens):
                if token in entity_text or entity_text in token:
                    ner_tags[i] = entity_label

        ner_extracted_data.append({
            'text': text,
            # 'tokens': tokens,
            'ner_tags': ner_tags,
            "ner_results": ner_results
        })

        print(f"Extracted NER for {len(translated_data[0])} examples in {split_name}")



    return ner_extracted_data

# Extract NER from both splits
# print("Extracting NER from train split...")
# ner_train = extract_ner_with_multilingual_model('train')

print("Extracting NER from test split...")
ner_test = extract_ner_with_multilingual_model('test')

# Create dataset with NER extracted
ner_extracted_dataset = DatasetDict({
    # 'train': Dataset.from_list(ner_train),
    'test': Dataset.from_list(ner_test)
})

print(f"\nNER extraction complete!")
# print(f"NER extracted train size: {len(ner_extracted_dataset['train'])}")
print(f"NER extracted test size: {len(ner_extracted_dataset['test'])}")

# Save NER extracted dataset
# ner_extracted_dataset['train'].to_json(pathTo('bable_ner_extracted_train.json'))
ner_extracted_dataset['test'].to_json(pathTo('bable_ner_extracted_test.json'))

In [ ]:
# Don't run this unless you want to translate with llama

def translate_with_llama32(text, source_lang='es', target_lang='en'):
    model = llama_32
    tokenizer = llama_32_tokenizer
    """Translate text using Llama-3.2-3B"""
    # This is a common pattern for instruction-following models.
    prompt = f"Translate the following {source_lang} text to {target_lang}. Only provide the translation, no explanations, keep entities names as is, whether it's names, tools:\n\n{text}, Translation:"


    messages = [
        # {
        #     "role": "system",
        #     "content": "You are a helpful AI assistant. Answer the user's questions clearly and concisely."
        # },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 4. Format Input with Chat Template
    # This applies the special tokens like <|start_header_id|> automatically
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)



    # 6. Generate Output with Ollama-equivalent Parameters
    outputs = model.generate(
        input_ids,
        max_new_tokens=2048,      # Limits response length
        temperature=0.8,          # Controls creativity (Ollama default ~0.8)
        top_p=0.9,                # Nucleus sampling
        top_k=40,                 # Limits vocabulary choices
        repetition_penalty=1.1,   # Prevents repeating phrases
        do_sample=True,           # Required for temp/top_p to work
        # streamer=streamer         # Print to console as it generates
    )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

    print(prompt)

    output = llama_32_pipeline(prompt)
    after_translation = output[0]['generated_text'].split('Translation:')[1]
    return after_translation


    processed = llama_32_processor.apply_chat_template(messages,
    add_generation_prompt=True, tokenize=False)
    inputs = llama_32_processor(processed, return_tensors="pt")


    # inputs = llama_32_tokenizer(prompt, return_tensors="pt").to(llama_32.device)

    outputs = llama_32.generate(
        **inputs,
        max_new_tokens=1000,
        # temperature=0.3,
        # do_sample=True,
        # pad_token_id=tokenizer.eos_token_id
    )

    translation = llama_32_processor.decode(outputs[0], skip_special_tokens=True)

    return translation

# Assuming sampleCV_ES is already defined from cell 3-3j1exer-Ft
print("Translating sampleCV_ES using Llama-3.2-3B...")
sampleCV_EN_FROM_ES_LLAMA32 = translate_with_llama32(sampleCV_ES, source_lang='spanish', target_lang='english')


In [ ]:








## RUN THIS ONLY IF YOU WANT TO GENERATE THE LLAMA_TRANSLATED_FILE










def translate_back_to_english_with_llama(text, source_lang='es'):
    return translate_with_llama32(text, 'spanish', 'english')


def translate_dataset_back(split_name):
    """Translate a dataset split back to English using Llama"""
    back_translated_data = []
    translated_data = translated_dataset[split_name]

    for idx in (translated_data[0]):
        example = translated_data[0][idx]
        print(example)

        # Join tokens to form text
        # text = ' '.join(tokens)


        # Translate back to English
        english_text = translate_with_llama32(example, 'spanish', 'english')

        print(english_text)

        back_translated_data.append({"en": english_text, "es":example  })

    return back_translated_data

# Back-translate both splits
# print("Back-translating train split...")
# back_translated_train = translate_dataset_back('train')

print("Back-translating test split...")
back_translated_test = translate_dataset_back('test')

# Create back-translated dataset
back_translated_dataset = DatasetDict({
    # 'train': Dataset.from_list(back_translated_train),
    'test': Dataset.from_list(back_translated_test)
})

print(f"\nBack-translation complete!")
# print(f"Back-translated train size: {len(back_translated_dataset['train'])}")
print(f"Back-translated test size: {len(back_translated_dataset['test'])}")

# Save back-translated dataset
# Write back_translated_test to file
with open(pathTo('llama_back_translated_test.json'), "w") as final:
	json.dump(back_translated_test, final)

In [ ]:
sampleCV_EN_FROM_ES_LLAMA = translate_back_to_english_with_llama(sampleCV_ES, 'es')
print(sampleCV_EN_FROM_ES_LLAMA)

In [ ]:
### READ FROM FILE
back_translated_test = json.load(open(pathTo('llama_back_translated_test.json'), 'r'))

In [ ]:
# bilora From LLama back translated to predictions

import json
from datasets import Dataset

# Load the raw back-translated texts from the file
# Assuming pathTo is defined in a previous cell
with open(pathTo('llama_back_translated_test.json'), 'r') as f:
    llama_raw_texts = json.load(f)

# Convert the list of strings into a Dataset object
# Each string will be a 'text' feature in the dataset
llama_dataset_raw = Dataset.from_list([{'text': text['en']} for text in llama_raw_texts])


# Tokenize the new dataset
tokenized_llama_ds = llama_dataset_raw.map(tokenize_raw_text, batched=True)

print("Tokenized Llama translated dataset created:")
print(tokenized_llama_ds)
with torch.autocast("cuda"):

  # --- Run predictions on the test dataset ---
  results = trainer.predict(tokenized_llama_ds)


predictions, label_ids, metrics = results.predictions, results.label_ids, results.metrics
print(label_ids)
# Convert raw logits to label indices.
predictions = np.argmax(predictions, axis=2)
print(predictions)


    # predictions = torch.argmax(outputs.logits, axis=2)
print(predictions.shape)
file_content = []
for idx, sample in enumerate(predictions):
  print(idx)
  ners = []
  for i in sample:
    # print(i)
    ners.append(id2label[i.item()])
  final_output = {
    "ners": ners,
    "original": ' '.join(dataset["test"][idx]["tokens"]),
    "back-translated": llama_raw_texts[idx],
    "mappings": (tokenized_llama_ds[idx]['offset_mapping'])
  }
  file_content.append(final_output)
# Write to file
with open(pathTo('predictions_bilora_from_llama_translated.json'), 'w') as f:
  json.dump(file_content, f)

    # Show top 3 ids and top


# Final


In [ ]:
# Predictions from Spanish directly
import json
from datasets import Dataset

# Load the raw back-translated texts from the file
# Assuming pathTo is defined in a previous cell
with open(pathTo('translated_test.json'), 'r') as f:
    llama_raw_texts = json.load(f)


print(llama_raw_texts[0])
# Convert the list of strings into a Dataset object
# Each string will be a 'text' feature in the dataset
llama_dataset_raw = Dataset.from_list([{'text': text} for text in llama_raw_texts])


# Tokenize the new dataset
tokenized_llama_ds = llama_dataset_raw.map(tokenize_raw_text, batched=True)

print("Tokenized Llama translated dataset created:")
print(tokenized_llama_ds)



with torch.autocast("cuda"):

  # --- Run predictions on the test dataset ---
  results = trainer.predict(tokenized_llama_ds)





predictions, label_ids, metrics = results.predictions, results.label_ids, results.metrics
print(label_ids)
# Convert raw logits to label indices.
predictions = np.argmax(predictions, axis=2)
print(predictions)


    # predictions = torch.argmax(outputs.logits, axis=2)
print(predictions.shape)
file_content = []
for idx, sample in enumerate(predictions):
  print(idx)
  ners = []
  for i in sample:
    # print(i)
    ners.append(id2label[i.item()])
  final_output = {
    "text": ' '.join(dataset["test"][idx]["tokens"]),
    "spanish": llama_raw_texts[idx],
    "ners": ners,
    "tokens": dataset["test"][idx]["tokens"],
    "mappings": (tokenized_llama_ds[idx]['offset_mapping'])
  }
  file_content.append(final_output)
# Write to file
with open(pathTo('predictions_bilora_from_spanish_directly.json'), 'w') as f:
  json.dump(file_content, f)

    # Show top 3 ids and top


# Final



```
Before extending max_length:
Test Evaluation Metrics:
test_loss: 0.2483
test_precision: 0.6681
test_recall: 0.6923
test_f1: 0.6800
test_accuracy: 0.9182
test_runtime: 2.7690
test_samples_per_second: 18.0570
test_steps_per_second: 2.5280

After Extending max_length:
test_loss: 0.2222
test_precision: 0.7371
test_recall: 0.7035
test_f1: 0.7199
test_accuracy: 0.9323
test_runtime: 10.7508
test_samples_per_second: 4.6510
test_steps_per_second: 2.3250


After Extending max_length 20 epochs:
Train:
Test Evaluation Metrics:
test_loss: 0.0645
test_precision: 0.8603
test_recall: 0.8663
test_f1: 0.8633
test_accuracy: 0.9763
test_runtime: 122.6254
test_samples_per_second: 4.4440
test_steps_per_second: 2.2260

Test:
test_loss: 0.3263
test_precision: 0.7289
test_recall: 0.6650
test_f1: 0.6955
test_accuracy: 0.9264
test_runtime: 10.7205
test_samples_per_second: 4.6640
test_steps_per_second: 2.3320




After one go 20 epochs:
Test Evaluation Metrics:
test_loss: 0.2562
test_precision: 0.7249
test_recall: 0.6414
test_f1: 0.6806
test_accuracy: 0.9225
test_runtime: 10.8974
test_samples_per_second: 4.5880
test_steps_per_second: 2.2940
```



In [ ]:
# --- Analyze Incorrect Findings per Label ---

from collections import defaultdict

# Initialize dictionaries to hold false positives and false negatives for each label.
false_positives = defaultdict(list)
false_negatives = defaultdict(list)

# Assume that the test dataset contains the original token texts under the "tokens" field.
# Adjust this if your dataset uses a different key.
test_tokens = tokenized_ds["test"]["tokens"]

# Iterate over each test example.
for example_idx, (pred_seq, true_seq, tokens) in enumerate(zip(true_predictions, true_labels, test_tokens)):
    for token_idx, (pred_label, true_label, token) in enumerate(zip(pred_seq, true_seq, tokens)):
        if pred_label != true_label:
            # Record a false positive for the predicted label.
            false_positives[pred_label].append({
                "example_index": example_idx,
                "token_index": token_idx,
                "token": token,
                "true_label": true_label
            })
            # Record a false negative for the true label.
            false_negatives[true_label].append({
                "example_index": example_idx,
                "token_index": token_idx,
                "token": token,
                "predicted_label": pred_label
            })

# For each label, print the incorrect findings along with false positives and false negatives.
print("\nDetailed Error Analysis per Label:")
for label in label_list:
    if label == 'O':
      continue
    if label !='Email Address':
      continue
    print(f"\n=== Label: {label} ===")

    # Incorrect findings are the union of false positives and false negatives for this label.
    fp = false_positives.get(label, [])
    fn = false_negatives.get(label, [])

    print(f"\nFalse Positives for '{label}' (predicted as {label} but true label different) [{len(fp)}]:")
    if fp:
        for error in fp:
            print(f"  - Ex {error['example_index']} Token {error['token_index']}: '{error['token']}' "
                  f"(predicted: {label}, true: {error['true_label']})")
    else:
        print("  None.")

    print(f"\nFalse Negatives for '{label}' (true label is {label} but predicted differently) [{len(fn)}]:")
    if fn:
        for error in fn:
            print(f"  - Ex {error['example_index']} Token {error['token_index']}: '{error['token']}' "
                  f"(true: {label}, predicted: {error['predicted_label']})")
    else:
        print("  None.")



In [ ]:
# Print accuracy per element in test dataset
print("\nAccuracy per Element in Test Dataset:")
for example_idx, (pred_seq, true_seq) in enumerate(zip(true_predictions, true_labels)):
    correct_count = sum(1 for pred, true in zip(pred_seq, true_seq) if pred == true and true !='O')
    total_count = len(pred_seq) - sum(1 for true in true_seq if true == 'O')
    accuracy = correct_count / total_count
    print(f"Example {example_idx + 1}: Accuracy = {accuracy:.4f}: Length = {len(dataset['train'][example_idx]['tokens'])}")

In [ ]:
# --- Display a few example predictions ---
print("\nSample Predictions:")
num_examples_to_show = 1 # Adjust as needed
for i in range(num_examples_to_show):
    # Retrieve the original tokens from the test split. Depending on your mapping,
    # the original 'tokens' field might still be available.
    tokens = dataset["test"][i]["tokens"]
    print("-" * 50)
    print(f"Example {i+1}:")
    # print("Tokens:         ", tokens)
    # print("Predicted Tags: ", true_predictions[i])
    # print("True Tags:      ", true_labels[i])
    for j in range(len(tokens)):
      token = tokens[j]
      true_label = true_labels[i][j]
      prediction = true_predictions[i][j]
      if(true_label=='O'):
        # print(token, end=' ')
        if(prediction !=true_label):
          print(f"(Wrong {prediction})")
      else:
        if(prediction !=true_label):
          print(f"{token}({prediction} should be {true_label})")
          if(true_label=='Email Address'):
            print("-" * 50)
            print(f"{token} ({true_label}, correct)")
            print("-" * 50)

        else:
          print(f"{token} ({true_label}, correct)")

    print("-" * 50)
    print("-" * 50)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# --- First, collect ALL true labels and predictions ---
all_true_labels = []
all_predictions = []

for i in range(len(dataset["test"])):
    for true_label, prediction in zip(true_labels[i], true_predictions[i]):
        all_true_labels.append(true_label)
        all_predictions.append(prediction)

# --- Create confusion matrix ---
labels = sorted(list(set(all_true_labels + all_predictions)))  # Get unique sorted labels
cm = confusion_matrix(all_true_labels, all_predictions, labels=labels)


cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize each row


# --- Plotting ---
plt.figure(figsize=(15, 12))
sns.heatmap(cm_normalized,     annot=cm,  # Show absolute counts as annotations
 fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# --- Compute classification report ---
report = classification_report(
    all_true_labels,
    all_predictions,
    target_names=labels,
    output_dict=True  # Return as dictionary for easier processing
)



# --- Convert to DataFrame and format ---
report_df = pd.DataFrame(report).transpose()
report_df = report_df.round(2)  # Round to 2 decimals
report_df = report_df.drop(columns=['support'])  # Optional: Remove support column

# --- Print with clear formatting ---
print("\nPer-Class Metrics:")
print(report_df.to_markdown(tablefmt="grid", floatfmt=".2f"))

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import pandas as pd

# --- Compute metrics ---
report = classification_report(
    all_true_labels,
    all_predictions,
    target_names=labels,
    output_dict=True,
    zero_division=0  # Handle classes with no predictions
)

# --- Calculate micro-average metrics ---
micro_precision = precision_score(all_true_labels, all_predictions, average='micro', zero_division=0)
micro_recall = recall_score(all_true_labels, all_predictions, average='micro', zero_division=0)
micro_f1 = f1_score(all_true_labels, all_predictions, average='micro', zero_division=0)

# --- Add micro-average to the report ---
report["micro avg"] = {
    "precision": micro_precision,
    "recall": micro_recall,
    "f1-score": micro_f1,
    "support": report["weighted avg"]["support"]  # Total number of samples
}

# --- Calculate macro-average metrics ---
micro_precision = precision_score(all_true_labels, all_predictions, average='macro', zero_division=0)
micro_recall = recall_score(all_true_labels, all_predictions, average='macro', zero_division=0)
micro_f1 = f1_score(all_true_labels, all_predictions, average='macro', zero_division=0)

# --- Add micro-average to the report ---
report["macro avg"] = {
    "precision": micro_precision,
    "recall": micro_recall,
    "f1-score": micro_f1,
    "support": report["weighted avg"]["support"]  # Total number of samples
}

# --- Remove macro-average (optional) ---
# del report["macro avg"]

# --- Convert to DataFrame and format ---
report_df = pd.DataFrame(report).transpose().round(2)
report_df = report_df.drop(columns=["support"])  # Optional: Remove support column

print("\nPer-Class Metrics with Micro Average:")
print(report_df.to_markdown(tablefmt="grid", floatfmt=".2f"))

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import pandas as pd



# Remove all 'O'
all_true_labels_without_o = []
all_predictions_without_o = []
for index, label in enumerate(all_true_labels):
  if label!='O' and all_predictions[index] !='O':
    all_true_labels_without_o.append(label)
    all_predictions_without_o.append(all_predictions[index])
labels_without_o = labels.copy()
labels_without_o.remove('O')
# --- Compute metrics ---
report2 = classification_report(
    all_true_labels_without_o,
    all_predictions_without_o,
    target_names=labels_without_o,
    output_dict=True,
    zero_division=0  # Handle classes with no predictions
)




# --- Calculate micro-average metrics ---
micro_precision = precision_score(all_true_labels_without_o, all_predictions_without_o, average='micro', zero_division=0)
micro_recall = recall_score(all_true_labels_without_o, all_predictions_without_o, average='micro', zero_division=0)
micro_f1 = f1_score(all_true_labels_without_o, all_predictions_without_o, average='micro', zero_division=0)

# --- Add micro-average to the report ---
report2["micro avg"] = {
    "precision": micro_precision,
    "recall": micro_recall,
    "f1-score": micro_f1,
    "support": report["weighted avg"]["support"]  # Total number of samples
}

# --- Calculate macro-average metrics ---
# macro_precision = precision_score(all_true_labels_without_o, all_predictions_without_o, average='macro', zero_division=0)
# macro_recall = recall_score(all_true_labels_without_o, all_predictions_without_o, average='macro', zero_division=0)
# macro_f1 = f1_score(all_true_labels_without_o, all_predictions_without_o, average='macro', zero_division=0)

# --- Add macro-average to the report ---
# report["macro avg"] = {
#     "precision": macro_precision,
#     "recall": macro_recall,
#     "f1-score": macro_f1,
#     "support": report["weighted avg"]["support"]  # Total number of samples
# }

# --- Remove macro-average (optional) ---
# del report["macro avg"]

# --- Convert to DataFrame and format ---
report_df = pd.DataFrame(report2).transpose().round(2)
report_df = report_df.drop(columns=["support"])  # Optional: Remove support column

print("\nPer-Class Metrics with Micro Average:")
print(report_df.to_markdown(tablefmt="grid", floatfmt=".2f"))

In [ ]:
print(tokenized_ds["test"][0]['tokens'])

In [ ]:
print(dataset['test'])

In [ ]:
sample_dataset = Dataset.from_list([{'tokens': [words]}])
def align_sample(examples):
      tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, padding='longest',
      max_length=MAX_LENGTH,
                                   return_tensors="pt",
      truncation=True).to('cuda')
      return tokenized_inputs


tokenized_sample = sample_dataset.map(align_sample)
# print(tokenized_inputs['input_ids'][0])
print(tokenized_sample['input_ids'][0][0])
# print(**tokenized_sample['input_ids'][0][0])
# Predict
predictedSample = trainer.predict([{'input_ids':tokenized_sample['input_ids'][0][0]}])
# predicatedSample = bilora_model(**tokenized_sample['input_ids'][0][0])
sample_predictions, sample_label_ids, sample_metrics = predictedSample.predictions, predictedSample.label_ids, predictedSample.metrics

# Convert raw logits to label indices.
sample_predictions = np.argmax(sample_predictions, axis=2)

for i,p in zip(tokenized_sample['input_ids'][0][0],sample_predictions[0]):
  print(tokenizer.decode(i),id2label[p])

In [ ]:
print(label2id)
print(id2label)
print(label_list)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "false"
os.environ["WANDB_API_KEY"]="REDACTED"
import wandb
run = wandb.init(project="my_project")

with torch.autocast("cuda"):
  sample_dataset = Dataset.from_list([{'tokens': [words]}])
  def align_sample(examples):
        tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, padding='longest', max_length=MAX_LENGTH, truncation=True)
        return tokenized_inputs


  tokenized_sample = sample_dataset.map(align_sample)

  # Predict
  predictedSample = trainer.predict([{'input_ids':tokenized_sample['input_ids'][0][0]}])

  sample_predictions, sample_label_ids, sample_metrics = predictedSample.predictions, predictedSample.label_ids, predictedSample.metrics

  # Convert raw logits to label indices.
  sample_predictions = np.argmax(sample_predictions, axis=2)

  for i,p in zip(tokenized_sample['input_ids'][0][0],sample_predictions[0]):
    print(tokenizer.decode(i),id2label[p])